# Exercise A: Train Your First Object Detector

Back to the course site: https://neel429.github.io/ml-for-robotics-/

This notebook trains YOLO26n on the dataset you exported from Roboflow.

## Cell 1 - Install dependencies

The `-q` flag keeps the install log shorter.

In [ ]:
!pip install ultralytics roboflow -q

## Cell 2 - Check GPU

If this prints `None`, switch Colab to **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None - check runtime type'}")

## Cell 3 - Download your dataset from Roboflow

Replace this example with the exact snippet from your Roboflow export dialog.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_KEY_HERE")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("yolov8")

## Cell 4 - Inspect the dataset structure

Update `dataset_path` to the folder Roboflow created.

In [ ]:
import os

dataset_path = "YOUR-PROJECT-NAME-1"  # folder Roboflow created
for split in ["train", "valid", "test"]:
    images = len(os.listdir(f"{dataset_path}/{split}/images"))
    labels = len(os.listdir(f"{dataset_path}/{split}/labels"))
    print(f"{split}: {images} images, {labels} labels")

## Cell 5 - Look at one label file

YOLO labels use: `class_id x_center y_center width height`, normalized from 0 to 1.

In [ ]:
import random

label_files = os.listdir(f"{dataset_path}/train/labels")
sample = random.choice(label_files)
with open(f"{dataset_path}/train/labels/{sample}") as f:
    content = f.read()
print(f"File: {sample}")
print(f"Contents:\n{content}")
print("\nFormat: class_id  x_center  y_center  width  height (all normalized 0-1)")

## Cell 6 - Visualize a training image with labels

Always verify boxes before training.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

img_dir = f"{dataset_path}/train/images"
lbl_dir = f"{dataset_path}/train/labels"

img_file = random.choice(os.listdir(img_dir))
lbl_file = img_file.replace(".jpg", ".txt").replace(".png", ".txt")

img = cv2.imread(f"{img_dir}/{img_file}")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

with open(f"{lbl_dir}/{lbl_file}") as f:
    for line in f:
        parts = list(map(float, line.strip().split()))
        cx, cy, bw, bh = parts[1]*w, parts[2]*h, parts[3]*w, parts[4]*h
        x1, y1 = int(cx - bw/2), int(cy - bh/2)
        x2, y2 = int(cx + bw/2), int(cy + bh/2)
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)

plt.imshow(img)
plt.title(f"Sample: {img_file}")
plt.axis("off")
plt.show()

## Cell 7 - Train YOLO26n

Reduce `batch` to 8 if Colab runs out of GPU memory.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")   # start from pretrained nano weights

results = model.train(
    data=f"{dataset_path}/data.yaml",
    epochs=50,
    imgsz=416,
    batch=16,
    name="my_detector",
    patience=15,          # stop early if val loss stops improving
    save=True,
    device=0              # GPU device 0 - auto-falls back to CPU
)

## Cell 8 - Find the trained weights

Download `best.pt` and place it in your local `my-detector` folder.

In [ ]:
import glob

weights = glob.glob("runs/detect/my_detector/weights/best.pt")
print(f"Best weights saved at: {weights[0]}")